## XLM-RoBERTa BIO token classification

notebook for trainig a multilingual RoBERTa-style model (`xlm-roberta-base`) for address parsing using BIO tags

In [ ]:
!pip install --upgrade transformers scipy scikit-learn numpy rapidfuzz optuna

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
# from pathlib import Path

# sys.path.append(str(Path().resolve().parent))

project_root = '/content/drive/MyDrive/TAI/sem3/NLP'

if project_root not in sys.path:
    sys.path.append(project_root)

%cd {project_root}/methods

In [ ]:
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

from sklearn.metrics import accuracy_score, f1_score

from data_preparation.labels import align_labels, bio_to_dict
from data_preparation.preprocess_data import evaluate_predictions, make_splits, preprocess_data

### Define constants and load data

In [ ]:
MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 128
SEED = 42
N_FOLDS = 3
N_TRIALS = 8

FIELDS = ["house_number", "street", "city", "postal_code", "state", "country"]
BIO_LABELS = ["O"] + [f"B-{f}" for f in FIELDS] + [f"I-{f}" for f in FIELDS]
label2id = {label: idx for idx, label in enumerate(BIO_LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

set_seed(SEED)

In [ ]:
df = preprocess_data()
print(f"Loaded {len(df)} rows from dataset.")

### Define helper functions

In [ ]:
def build_model() -> AutoModelForTokenClassification:
    """
    Initializes and returns a pre-trained token classification model.
    
    Returns:
        AutoModelForTokenClassification: The instantiated Hugging Face model
        configured with the appropriate number of labels and label mappings.
    """
    return AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(BIO_LABELS),
        label2id=label2id,
        id2label=id2label,
    )


def build_tokenized_dataset(
    split_df: pd.DataFrame, tokenizer: AutoTokenizer
) -> Tuple[Dataset, Dict[str, List[Any]]]:
    """
    Constructs a tokenized Hugging Face Dataset from a pandas DataFrame.
    
    Args:
        split_df (pd.DataFrame): DataFrame containing input and target columns.
        tokenizer (AutoTokenizer): The tokenizer to use for encoding the text.
        
    Returns:
        Tuple[Dataset, Dict[str, List[Any]]]: A tuple containing:
            - The tokenized Hugging Face Dataset with input_ids, attention_mask, labels, and word_ids.
            - A metadata dictionary containing original tokens and gold targets.
    """
    input_ids_all: List[List[int]] = []
    attention_all: List[List[int]] = []
    labels_all: List[List[int]] = []
    word_ids_all: List[List[int]] = []

    meta_tokens: List[List[str]] = []
    meta_gold_targets: List[Dict[str, str]] = []

    for _, row in split_df.iterrows():
        tokens, bio_labels = align_labels(str(row["input"]), row["target"])
        if not tokens:
            continue

        encoded = tokenizer(
            list(tokens),
            is_split_into_words=True,
            truncation=True,
            max_length=MAX_LEN,
        )
        word_ids = encoded.word_ids()

        label_ids: List[int] = []
        previous_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(label2id.get(bio_labels[word_id], label2id["O"]))
            else:
                label_ids.append(-100)
            previous_word_id = word_id

        input_ids_all.append(encoded["input_ids"])
        attention_all.append(encoded["attention_mask"])
        labels_all.append(label_ids)
        word_ids_all.append([-1 if w is None else int(w) for w in word_ids])

        meta_tokens.append(list(tokens))
        meta_gold_targets.append(row["target"])

    hf_ds = Dataset.from_dict(
        {
            "input_ids": input_ids_all,
            "attention_mask": attention_all,
            "labels": labels_all,
            "word_ids": word_ids_all,
        }
    )

    metadata = {
        "tokens": meta_tokens,
        "gold_targets": meta_gold_targets,
    }
    return hf_ds, metadata


def compute_metrics(eval_preds: Tuple[np.ndarray, np.ndarray]) -> Dict[str, float]:
    """
    Computes evaluation metrics (weighted F1 and accuracy) for token classification.
    
    Args:
        eval_preds (Tuple[np.ndarray, np.ndarray]): A tuple of (logits, labels) from the model.
        
    Returns:
        Dict[str, float]: A dictionary containing f1_weighted and accuracy scores.
    """
    logits, labels = eval_preds
    preds = np.argmax(logits, axis=-1)

    y_true: List[int] = []
    y_pred: List[int] = []
    for pred_row, label_row in zip(preds, labels):
        for pred_id, label_id in zip(pred_row, label_row):
            if label_id == -100:
                continue
            y_true.append(int(label_id))
            y_pred.append(int(pred_id))

    if not y_true:
        return {"f1_weighted": 0.0, "accuracy": 0.0}

    return {
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }


def optuna_hp_space(trial: Any) -> Dict[str, Any]:
    """
    Defines the hyperparameter search space for Optuna.
    
    Args:
        trial (Any): An Optuna trial object.
        
    Returns:
        Dict[str, Any]: A dictionary of hyperparameter names and their sampled values.
    """
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
        "warmup_steps": trial.suggest_int("warmup_steps", 0, 500),
        "num_train_epochs": trial.suggest_categorical("num_train_epochs", [2, 3]),
    }


def decode_predictions_to_dicts(
    prediction_logits: np.ndarray,
    metadata: Dict[str, List[Any]],
) -> List[Dict[str, str]]:
    """
    Converts model logits back into structured dictionaries of address fields.
    
    Args:
        prediction_logits (np.ndarray): The raw logits output by the model.
        metadata (Dict[str, List[Any]]): Metadata containing word_ids and tokens for each example.
            
    Returns:
        List[Dict[str, str]]: A list of dictionaries, where each dictionary maps
            address field names to their predicted string values.
    """
    pred_ids = np.argmax(prediction_logits, axis=-1)
    pred_dicts: List[Dict[str, str]] = []

    for example_idx, example_pred_ids in enumerate(pred_ids):
        word_ids = metadata["word_ids"][example_idx]
        tokens = metadata["tokens"][example_idx]

        seen_word_ids = set()
        word_level_labels: List[str] = []
        active_word_count = 0

        for token_idx, word_id in enumerate(word_ids):
            if word_id == -1 or word_id in seen_word_ids:
                continue
            seen_word_ids.add(word_id)
            active_word_count = max(active_word_count, word_id + 1)
            label_name = id2label.get(int(example_pred_ids[token_idx]), "O")
            word_level_labels.append(label_name)

        used_tokens = tuple(tokens[:active_word_count])
        pred_dicts.append(bio_to_dict(used_tokens, word_level_labels))

    return pred_dicts

### Hyperparameter search using Optuna

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Fold-0 hyperparameter search (weighted F1)
first_split = next(make_splits(df))
train_ds_0, _ = build_tokenized_dataset(first_split["train"], tokenizer)
val_ds_0, _ = build_tokenized_dataset(first_split["val"], tokenizer)

def model_init():
    """
    Initializes the model for hyperparameter search.
    
    Returns:
        AutoModelForTokenClassification: A fresh instance of the model.
    """
    return build_model()

hp_args = TrainingArguments(
    output_dir="./outputs/roberta/hp_search_fold0",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=0,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    seed=SEED,
    report_to=[],
)

hp_trainer = Trainer(
    model_init=model_init,
    args=hp_args,
    train_dataset=train_ds_0,
    eval_dataset=val_ds_0,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

best_run = hp_trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    n_trials=N_TRIALS,
    hp_space=optuna_hp_space,
    compute_objective=lambda metrics: metrics["eval_f1_weighted"],
)

best_hyperparameters = best_run.hyperparameters
print("Best HPs from fold-0 search:")
print(best_hyperparameters)

### Train and evaluate across 3 folds

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

fold_summaries = []

for fold_idx, split in enumerate(make_splits(df)):
    print('=' * 60)
    print(f"FOLD {fold_idx + 1} / {N_FOLDS}")
    print(f"train={len(split['train'])} val={len(split['val'])} test={len(split['test'])}")
    print('=' * 60)

    train_ds, _ = build_tokenized_dataset(split["train"], tokenizer)
    val_ds, _ = build_tokenized_dataset(split["val"], tokenizer)
    test_ds, test_meta = build_tokenized_dataset(split["test"], tokenizer)
    test_meta["word_ids"] = test_ds["word_ids"]

    model = build_model()

    fold_output_dir = f"./outputs/roberta/fold_{fold_idx + 1}"
    args = TrainingArguments(
        output_dir=fold_output_dir,
        num_train_epochs=int(best_hyperparameters.get("num_train_epochs", 3)),
        learning_rate=float(best_hyperparameters.get("learning_rate", 2e-5)),
        weight_decay=float(best_hyperparameters.get("weight_decay", 0.01)),
        warmup_steps=int(best_hyperparameters.get("warmup_steps", 0)),
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=1,
        logging_steps=50,
        seed=SEED + fold_idx,
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    test_output = trainer.predict(test_ds)
    pred_dicts = decode_predictions_to_dicts(test_output.predictions, test_meta)
    gold_dicts = test_meta["gold_targets"]

    struct_metrics = evaluate_predictions(gold_dicts, pred_dicts)
    token_metrics = compute_metrics((test_output.predictions, test_output.label_ids))

    print("Structured metrics on test:")
    print(f"  examples: {struct_metrics['counts']['examples']}")
    print(f"  exact_match: {struct_metrics['exact_match']:.4f}")
    print(f"  overall_item_accuracy: {struct_metrics['overall_item_accuracy']:.4f}")
    print("Token metrics on test:")
    print(f"  f1_weighted: {token_metrics['f1_weighted']:.4f}")
    print(f"  accuracy:    {token_metrics['accuracy']:.4f}")

    trainer.save_model(f"{fold_output_dir}/best_model")
    tokenizer.save_pretrained(f"{fold_output_dir}/best_model")

    summary_entry = {
        "fold": fold_idx + 1,
        "best_val_loss": trainer.state.best_metric,
        "test_examples": struct_metrics["counts"]["examples"],
        "test_exact_match": struct_metrics["exact_match"],
        "test_overall_item_accuracy": struct_metrics["overall_item_accuracy"],
        "test_token_f1_weighted": token_metrics["f1_weighted"],
        "test_token_accuracy": token_metrics["accuracy"],
    }
    for field in FIELDS:
        summary_entry[f"test_{field}_acc"] = struct_metrics["per_field_accuracy"][field]
    fold_summaries.append(summary_entry)

### Results summary

In [ ]:
summary_df = pd.DataFrame(fold_summaries)
print("=" * 120)
print("XLM-R BIO 3-FOLD SUMMARY")
print("=" * 120)
print(summary_df.to_string(index=False))

print("\nMean metrics across folds:")
mean_cols = [
    "best_val_loss",
    "test_exact_match",
    "test_overall_item_accuracy",
    "test_token_f1_weighted",
    "test_token_accuracy",
]
print(summary_df[mean_cols].mean().to_string())

metric_map = {
    "Exact Match (EM)": "test_exact_match",
    "Overall Item Accuracy": "test_overall_item_accuracy",
    "house_number": "test_house_number_acc",
    "street": "test_street_acc",
    "city": "test_city_acc",
    "country": "test_country_acc",
    "postal_code": "test_postal_code_acc",
    "state": "test_state_acc",
}

rows = []
for metric_name, col in metric_map.items():
    values = summary_df[col].astype(float).to_numpy()
    mean_pct = values.mean() * 100.0
    sd_pct = values.std(ddof=1) * 100.0 if len(values) > 1 else 0.0

    row = {
        "Metric [%]": metric_name,
        "Fold 0": values[0] * 100.0,
        "Fold 1": values[1] * 100.0,
        "Fold 2": values[2] * 100.0,
        "Average (± SD)": f"{mean_pct:.1f} ± {sd_pct:.1f}",
    }
    rows.append(row)

detailed_df = pd.DataFrame(rows)
print("\n" + "=" * 120)
print("DETAILED PERFORMANCE TABLE (PERCENT)")
print("=" * 120)
print(detailed_df.to_string(index=False, formatters={
    "Fold 0": "{:.1f}".format,
    "Fold 1": "{:.1f}".format,
    "Fold 2": "{:.1f}".format,
}))

# Additional: LaTeX-ready export for report tables
latex_df = detailed_df.copy()
for fold_col in ["Fold 0", "Fold 1", "Fold 2"]:
    latex_df[fold_col] = latex_df[fold_col].map(lambda x: f"{x:.1f}")

print("\nLaTeX table:\n")
print(latex_df.to_latex(index=False, escape=False, column_format="|l|c|c|c|c|"))